Name: Eileen Yang
<br>
Github Username: eileeny32
<br>
USC ID: 6974019134

In [1]:
import pandas as pd
import numpy as np
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as Pipe
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.cluster import KMeans

The following code block completes 1(a).

In [2]:
df = pd.read_csv('Frogs_MFCCs.csv')

rng = np.random.default_rng()
shuffle_idx = rng.permutation(df.shape[0])
split_idx = int(df.shape[0]*0.7)

train_df = df.iloc[shuffle_idx[:split_idx], :]
test_df = df.iloc[shuffle_idx[split_idx:], :]
y_train = train_df[['Family', 'Genus', 'Species']]
y_test = test_df[['Family', 'Genus', 'Species']]
X_train = train_df.drop(columns=['Family', 'Genus', 'Species', 'RecordID'])
X_test = test_df.drop(columns=['Family', 'Genus', 'Species', 'RecordID'])

The following paragraphs complete 1(b)(i).

For exact match, sklearn's accuracy_score metric does not support multiclass-multilabel classification; however, it is fairly simple to do the calculation by hand. The calculation itself is simply seeing if a row in y_pred perfectly matches its corresponding row in y_true; if so, that sample is assigned 1, else 0. Then the average over all samples is taken for the exact match score.

For Hamming loss, sklearn similarly does not support multiclass-multilabel classification for this metric. However, it is also simple to calculate as you are simply comparing each individual prediction in y_pred and comparing it to the corresponding individual prediction in y_true. Then, you divide the number of incorrect individual predictions by the total number of predictions (num samples\*num labels) to get the overall Hamming loss.  

In [19]:
def svm_pred(X_train, y_train, X_test, l1=False, smote=False):
  if l1:
    est = Pipeline([
        ('scale', StandardScaler()),
        ('model', OneVsRestClassifier(LinearSVC(penalty='l1', max_iter=5000))),
    ])
    param_grid = {
        'model__estimator__C': [0.0001, 0.001, 0.01, 0.1, 1]
    }
  elif smote:
    est = Pipe([
        ('scale', StandardScaler()),
        ('smote', SMOTE()),
        ('model', OneVsRestClassifier(LinearSVC(penalty='l1', max_iter=5000))),
    ])
    param_grid = {
        'model__estimator__C': [0.0001, 0.001, 0.01, 0.1, 1]
    }
  else:
    est = OneVsRestClassifier(SVC(kernel='rbf'))
    param_grid = {
      'estimator__C': [0.0001, 0.001, 0.01, 0.1, 1],
      'estimator__gamma': ['scale',  'auto', 0.1, 1.1, 2.1]
    }

  cv = GridSearchCV(
      estimator=est,
      param_grid=param_grid,
      cv=10,
      scoring='accuracy'
  )
  cv.fit(X_train, y_train)

  if l1 or smote:
    best_C = cv.best_params_['model__estimator__C']
    final_est = OneVsRestClassifier(LinearSVC(penalty='l1', C=best_C, max_iter=5000))
    scale = StandardScaler()
    X_train = scale.fit_transform(X_train)
    X_test = scale.transform(X_test)
    if smote:
      smote = SMOTE()
      X_train, y_train = smote.fit_resample(X_train, y_train)
  else:
    best_C = cv.best_params_['estimator__C']
    best_gamma = cv.best_params_['estimator__gamma']
    final_est = OneVsRestClassifier(SVC(kernel='rbf', C=best_C, gamma=best_gamma))
  final_est.fit(X_train, y_train)
  y_pred = final_est.predict(X_test)
  return y_pred

def hamming_score(y_pred, y_true): # adapted from: https://wiki.cloudfactory.com/docs/mp-wiki/metrics/hamming-score
  acc_list = []
  for i in range(y_true.shape[0]):
    acc_list.append(np.mean(y_pred == y_true[i]))
  return np.mean(acc_list)

The following code block completes 1(b)(ii).

In [11]:
total_pred = []
for i in y_train.columns:
  y_pred = svm_pred(X_train, y_train[i], X_test)
  total_pred.append(y_pred)
total_pred = np.column_stack(total_pred) # convert to NumPy array where each list (label preds) is a column

svm_acc = np.all(total_pred == y_test, axis=1).mean()
svm_hamming = np.mean(total_pred != y_test)
print(f'Exact match score from Gaussian SVM (OVR strategy): {svm_acc}')
print(f'Hamming loss from Gaussian SVM (OVR strategy): {svm_hamming}')

Exact match score from Gaussian SVM (OVR strategy): 0.9865678554886521
Hamming loss from Gaussian SVM (OVR strategy): 0.009726725335803613


The following code block completes 1(b)(iii).

In [9]:
total_pred_l1 = []
for i in y_train.columns:
  y_pred = svm_pred(X_train, y_train[i], X_test, l1=True)
  total_pred_l1.append(y_pred)
total_pred_l1 = np.column_stack(total_pred_l1)

l1_acc = np.all(total_pred_l1 == y_test, axis=1).mean()
l1_hamming = np.mean(total_pred_l1 != y_test)
print(f'Exact match score from L1-penalized SVM: {l1_acc}')
print(f'Hamming loss from L1-penalized SVM: {l1_hamming}')

Exact match score from L1-penalized SVM: 0.9124594719777674
Hamming loss from L1-penalized SVM: 0.0555812876331635


The following code block completes 1(b)(iv).

As far as the results show, it seems like the Gaussian SVM produced the best results, which is likely because it is more flexible than a linear SVM and this data is almost certainly not linearly separable. What is interesting to me is that SMOTE made the linear SVM perform a little worse. I am not sure why this is, as the dataset is definitely imbalanced.

In [8]:
total_pred_smote = []
for i in y_train.columns:
  y_pred = svm_pred(X_train, y_train[i], X_test, smote=True)
  total_pred_smote.append(y_pred)
total_pred_smote = np.column_stack(total_pred_smote)

smote_acc = np.all(total_pred_smote == y_test, axis=1).mean()
smote_hamming = np.mean(total_pred_smote != y_test)
print(f'Exact match score from SMOTE-adjusted L1-penalized SVM: {smote_acc}')
print(f'Hamming loss from SMOTE-adjusted L1-penalized SVM: {smote_hamming}')

Exact match score from SMOTE-adjusted L1-penalized SVM: 0.8698471514590088
Hamming loss from SMOTE-adjusted L1-penalized SVM: 0.06978539447274973


The following code block completes problem 2. I am using the silhouette score method to determine optimal k.

In [18]:
# since KMeans uses a distance metric between each pair of points to calculate silhouette score
# can precompute one distance matrix to use in calculations every time
y = df[['Family', 'Genus', 'Species']]
X = df.drop(columns=['Family', 'Genus', 'Species', 'RecordID'])

distances = pairwise_distances(X)
np.fill_diagonal(distances, 0)

# run KMeans 50 times
hamming_scores = []
hamming_distances = []
hamming_losses = []
for i in range(50):
  best_k = 0
  best_score = 0
  # find k automatically
  for j in range(2, 51):
    kmeans = KMeans(n_clusters=j)
    y_pred = kmeans.fit_predict(X)
    silhouette = silhouette_score(distances, y_pred, metric="precomputed")
    if silhouette > best_score:
      best_score = silhouette
      best_k = j
  final_kmeans = KMeans(n_clusters=best_k)
  final_y = final_kmeans.fit_predict(X)
  scores = []
  dists = []
  losses = []
  for j in np.unique(final_y):
    idxs = np.where(final_y == j)[0]
    cluster_label = []
    col_dists = []
    for k in y.columns:
      cluster = y[k].iloc[idxs]
      majority = y[k].iloc[idxs].value_counts().index[0]
      cluster_label.append(majority)
      col_dists.append(cluster[cluster != majority].shape[0])
    scores.append(hamming_score(np.array(cluster_label), cluster.to_numpy()))
    dists.append(sum(col_dists))
    losses.append(sum(col_dists)/(y.shape[0]*y.shape[1]))
  hamming_scores.append(np.mean(scores))
  hamming_distances.append(np.mean(dists))
  hamming_losses.append(np.mean(losses))
print(f'Average Hamming distance over 50 runs: {np.mean(hamming_distances)}')
print(f'SD of Hamming distances over 50 runs: {np.std(hamming_distances)}')

Average Hamming distance over 50 runs: 1344.1246666666668
SD of Hamming distances over 50 runs: 454.6072209946125


In [20]:
# additional averages and SD for Hamming scores and losses
print(f'Average Hamming scores over 50 runs: {np.mean(hamming_scores)}')
print(f'SD of Hamming scores over 50 runs: {np.std(hamming_scores)}')
print(f'Average Hamming losses over 50 runs: {np.mean(hamming_losses)}')
print(f'SD of Hamming losses over 50 runs: {np.std(hamming_losses)}')

Average Hamming scores over 50 runs: 0.2202492915420682
SD of Hamming scores over 50 runs: 0.022034173722807672
Average Hamming losses over 50 runs: 0.06227123774225928
SD of Hamming losses over 50 runs: 0.02106125647415393
